# 03 · Silver: SCD2 customers, open orders, delete detection
- Customers: SCD Type 2 rebuilt from Bronze (row hash + LAG to detect change, LEAD for end dates). Bad master data is flagged, not dropped.
- Open orders: a bad update is quarantined and the last good version is kept.
- Deletes: an order missing from the key snapshot is only marked **CANCELLED** after it is missing from **two consecutive** snapshots and has no invoice. A **snapshot guard** stops the run if the key list looks incomplete, so a broken extract can't mass-cancel orders.

In [ ]:
%run 00_common

In [ ]:
%%sql
CREATE OR REPLACE TABLE silver_customer AS
WITH typed AS (
  SELECT Company, No,
         NULLIF(TRIM(Name), '')  AS Name,
         Customer_Type,
         NULLIF(TRIM(ABN), '')   AS ABN,
         LOWER(TRIM(Email))      AS Email,
         City, State, Post_Code,
         Global_Dimension_1_Code AS Region_Code,
         Customer_Posting_Group,
         CAST(SystemModifiedAt AS TIMESTAMP) AS SystemModifiedAt,
         _batch_id,
         sha2(concat_ws('|', Name, Customer_Type, City, State, Global_Dimension_1_Code), 256) AS row_hash
  FROM bronze_customer
),
dedup AS (
  SELECT * FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY Company, No, SystemModifiedAt ORDER BY _batch_id DESC) AS rn
    FROM typed) WHERE rn = 1
),
changes AS (
  SELECT * FROM (
    SELECT *, LAG(row_hash) OVER (PARTITION BY Company, No ORDER BY SystemModifiedAt) AS prev_hash
    FROM dedup)
  WHERE prev_hash IS NULL OR prev_hash <> row_hash
),
chain AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY Company, No ORDER BY SystemModifiedAt) AS version_no,
         LEAD(SystemModifiedAt) OVER (PARTITION BY Company, No ORDER BY SystemModifiedAt) AS next_from
  FROM changes
)
SELECT Company, No, Name, Customer_Type, ABN, Email, City, State, Post_Code, Region_Code,
       Customer_Posting_Group, row_hash, version_no,
       CASE WHEN version_no = 1 THEN TIMESTAMP'1900-01-01' ELSE SystemModifiedAt END AS valid_from,
       COALESCE(next_from, TIMESTAMP'9999-12-31') AS valid_to,
       next_from IS NULL AS is_current,
       CASE WHEN Name IS NULL THEN 'Name is blank' END AS dq_issue,
       _batch_id
FROM chain

In [ ]:
%%sql
-- expect: 436 versions, 427 current, 9 changes, 1 flagged
SELECT COUNT(*) AS versions,
       SUM(CASE WHEN is_current THEN 1 ELSE 0 END)           AS current_customers,
       SUM(CASE WHEN version_no > 1 THEN 1 ELSE 0 END)       AS region_changes,
       SUM(CASE WHEN dq_issue IS NOT NULL THEN 1 ELSE 0 END) AS flagged
FROM silver_customer

In [ ]:
TS = "CAST(SystemModifiedAt AS TIMESTAMP) AS SystemModifiedAt"

load_silver("bronze_sales_header", "silver_sales_header", ["Company", "Document_Type", "No"], f"""
    Company, Document_Type, No, Sell_to_Customer_No, CAST(Order_Date AS DATE) AS Order_Date,
    Location_Code, Status, Salesperson_Code,
    Global_Dimension_1_Code AS Region_Code, Global_Dimension_2_Code AS Business_Unit,
    COALESCE(Delivery_Method, 'NOT_CAPTURED') AS Delivery_Method, {TS}""")

load_silver("bronze_sales_line", "silver_sales_line", ["Company", "Document_Type", "Document_No", "Line_No"], f"""
    Company, Document_Type, Document_No, CAST(Line_No AS INT) AS Line_No, No AS Item_No, Description,
    Location_Code,
    TRY_CAST(Quantity AS DECIMAL(18,4))   AS Quantity,
    TRY_CAST(Unit_Price AS DECIMAL(18,2)) AS Unit_Price,
    TRY_CAST(Amount AS DECIMAL(18,2))     AS Amount, {TS}""",
    rules=[("Quantity IS NULL", "Quantity blank or not numeric")])

In [ ]:
# ---- snapshot guard: never mass-cancel orders because of an incomplete key list ----
MIN_RATIO = 0.5   # today's snapshot must hold at least 50% of yesterday's keys

snaps = spark.sql("""SELECT _batch_id, COUNT(*) AS n FROM bronze_sales_header_keys
                     GROUP BY _batch_id ORDER BY _batch_id DESC LIMIT 2""").collect()
if not snaps or snaps[0].n == 0:
    raise Exception("Snapshot guard: latest key snapshot is missing or empty - not detecting deletes")
latest = snaps[0]._batch_id
prev = snaps[1]._batch_id if len(snaps) > 1 else ""
if len(snaps) > 1 and snaps[0].n < MIN_RATIO * snaps[1].n:
    raise Exception(f"Snapshot guard: {latest} has {snaps[0].n} keys vs {snaps[1].n} in {prev} - "
                    "looks incomplete, not detecting deletes")
print(f"Snapshot guard passed: {latest} = {snaps[0].n} keys, previous {prev} = {snaps[1].n if prev else 'n/a'}")

# ---- delete detection: two consecutive misses + invoice check ----
spark.sql(f"""
CREATE OR REPLACE TABLE silver_sales_order_status AS
WITH latest_keys AS (SELECT DISTINCT Company, No FROM bronze_sales_header_keys WHERE _batch_id = '{latest}'),
     prev_keys   AS (SELECT DISTINCT Company, No FROM bronze_sales_header_keys WHERE _batch_id = '{prev}'),
     posted      AS (SELECT DISTINCT Company, Order_No FROM silver_sales_invoice_header)
SELECT h.Company, h.No AS Order_No,
       CASE WHEN k.No  IS NOT NULL      THEN 'OPEN'
            WHEN p.Order_No IS NOT NULL THEN 'POSTED'          -- became an invoice: a real sale
            WHEN pk.No IS NOT NULL      THEN 'MISSING_ONCE'    -- wait one more snapshot before cancelling
            ELSE 'CANCELLED' END        AS order_status,
       k.No IS NOT NULL                 AS is_open,
       current_timestamp()              AS checked_at
FROM silver_sales_header h
LEFT JOIN latest_keys k ON k.Company = h.Company  AND k.No = h.No
LEFT JOIN prev_keys  pk ON pk.Company = h.Company AND pk.No = h.No
LEFT JOIN posted     p  ON p.Company = h.Company  AND p.Order_No = h.No
""")

In [ ]:
%%sql
-- expect: OPEN 29, POSTED 42, CANCELLED 6, MISSING_ONCE 3
SELECT order_status, COUNT(*) AS orders FROM silver_sales_order_status GROUP BY order_status